# Duplicates and Data Types in Pandas

Learn to identify duplicate records, choose keep rules, convert data types, parse dates, and validate cleaned data.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'customer_id': ['C01', 'C02', 'C02', 'C03'],
    'name': ['Asha', 'Ravi', 'Ravi', 'Meera'],
    'age': ['21', '34', '34', '29'],
    'joined': ['2025-01-10', '2025-02-12', '2025-02-12', '2025-03-01'],
    'active': ['True', 'False', 'False', 'True']
})
df

,customer_id,name,age,joined,active
0,C01,Asha,21,2025-01-10,True
1,C02,Ravi,34,2025-02-12,False
2,C02,Ravi,34,2025-02-12,False
3,C03,Meera,29,2025-03-01,True


## 1. Inspect data types

In [2]:
print(df.dtypes)
df.info()

customer_id    object
name           object
age            object
joined         object
active         object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  4 non-null      object
 1   name         4 non-null      object
 2   age          4 non-null      object
 3   joined       4 non-null      object
 4   active       4 non-null      object
dtypes: object(5)
memory usage: 292.0+ bytes


## 2. Detect duplicates

In [3]:
print('Duplicate rows:')
display(df[df.duplicated(keep=False)])
print('Duplicate customer IDs:')
display(df[df.duplicated(subset=['customer_id'], keep=False)])
print('Count of exact duplicate rows:', int(df.duplicated().sum()))

Duplicate rows:


,customer_id,name,age,joined,active
1,C02,Ravi,34,2025-02-12,False
2,C02,Ravi,34,2025-02-12,False


Duplicate customer IDs:


,customer_id,name,age,joined,active
1,C02,Ravi,34,2025-02-12,False
2,C02,Ravi,34,2025-02-12,False


Count of exact duplicate rows: 1


## 3. Remove duplicates

In [4]:
df.drop_duplicates()

,customer_id,name,age,joined,active
0,C01,Asha,21,2025-01-10,True
1,C02,Ravi,34,2025-02-12,False
3,C03,Meera,29,2025-03-01,True


In [5]:
df.drop_duplicates(subset=['customer_id'], keep='last')

,customer_id,name,age,joined,active
0,C01,Asha,21,2025-01-10,True
2,C02,Ravi,34,2025-02-12,False
3,C03,Meera,29,2025-03-01,True


## 4. Convert numeric and date columns

In [6]:
typed = df.copy()
typed['age'] = pd.to_numeric(typed['age'], errors='raise').astype('Int64')
typed['joined'] = pd.to_datetime(typed['joined'], errors='raise')
typed['name'] = typed['name'].astype('string')
typed.dtypes

customer_id            object
name           string[python]
age                     Int64
joined         datetime64[ns]
active                 object
dtype: object

## 5. Find values that fail conversion

In [7]:
raw_values = pd.Series(['1,250', '900', 'not known'])
clean_values = raw_values.str.replace(',', '', regex=False).str.strip()
numeric_values = pd.to_numeric(clean_values, errors='coerce')
print('Converted values:')
display(numeric_values)
print('Invalid source values:')
display(clean_values[numeric_values.isna() & clean_values.notna()])

Converted values:


0    1250.0
1     900.0
2       NaN
dtype: float64

Invalid source values:


2    not known
dtype: object

## 6. Convert Boolean labels explicitly

In [8]:
active = pd.Series(['True', 'False', 'true', 'false', 'unknown'])
active_bool = active.str.lower().map({'true': True, 'false': False}).astype('boolean')
pd.DataFrame({'source': active, 'converted': active_bool})

,source,converted
0,True,True
1,False,False
2,true,True
3,false,False
4,unknown,<NA>


## 7. Nullable types and category

In [9]:
ages = pd.Series([21, None, 29]).astype('Int64')
status = pd.Series(['new', 'active', 'new', 'closed']).astype('category')
print(ages, ages.dtype)
print(status.dtype)

0      21
1    <NA>
2      29
dtype: Int64 Int64
category


## 8. End-to-end validation

In [10]:
raw = df.copy()
cleaned = raw.sort_values('joined').drop_duplicates(subset=['customer_id'], keep='last').copy()
cleaned['customer_id'] = cleaned['customer_id'].astype('string')
cleaned['age'] = pd.to_numeric(cleaned['age'], errors='coerce').astype('Int64')
cleaned['joined'] = pd.to_datetime(cleaned['joined'], errors='coerce')
cleaned['active'] = cleaned['active'].str.lower().map({'true': True, 'false': False}).astype('boolean')
print('Raw row count:', len(raw), '| Cleaned row count:', len(cleaned))
print('Missing values:\n', cleaned.isna().sum())
print('Customer IDs unique:', cleaned['customer_id'].is_unique)
print('Final dtypes:\n', cleaned.dtypes)
cleaned

Raw row count: 4 | Cleaned row count: 3
Missing values:
 customer_id    0
name           0
age            0
joined         0
active         0
dtype: int64
Customer IDs unique: True
Final dtypes:
 customer_id    string[python]
name                   object
age                     Int64
joined         datetime64[ns]
active                boolean
dtype: object


,customer_id,name,age,joined,active
0,C01,Asha,21,2025-01-10,True
2,C02,Ravi,34,2025-02-12,False
3,C03,Meera,29,2025-03-01,True


## Key takeaways

- Exact duplicates and duplicate business keys are different concepts.
- Choose a deterministic keep rule before dropping records.
- Convert columns explicitly and review values that become missing.
- Validate uniqueness, null counts, row counts, and dtypes after cleaning.